<a href="https://colab.research.google.com/github/fabriciodansi-maker/academia/blob/main/academia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%%writefile main.py
from dataclasses import dataclass
from datetime import datetime, timedelta

class PlanoVencidoException(Exception):
    pass

class AlunoInativoException(Exception):
    pass

class AcessoNegadoException(Exception):
    pass

@dataclass
class Plano:
    nome: str
    valor: float
    duracao_dias: int

@dataclass
class Acesso:
    id_aluno: int
    data_hora: datetime
    tipo: str

class Pessoa:
    def __init__(self, id: int, nome: str, idade: int):
        self.id = id
        self.nome = nome
        self.idade = idade

    def exibir_dados(self):
        print(f"Pessoa: {self.nome}")

class Usuario(Pessoa):
    def __init__(self, id: int, nome: str, idade: int, login: str, senha: str, email: str):
        super().__init__(id, nome, idade)
        self.login = login
        self.senha = senha
        self.email = email

    def exibir_dados(self):
        print(f"Usuário: {self.nome} | Login: {self.login} | Email: {self.email}")

class Aluno(Usuario):
    def __init__(self, id: int, nome: str, idade: int, login: str, senha: str, plano: Plano):
        super().__init__(id, nome, idade, login, senha, "")
        self.plano = plano
        self.ativo = True
        self.data_vencimento = datetime.now() + timedelta(days=plano.duracao_dias)
        self.dentro = False

    def exibir_dados(self):
        print(f"Aluno ID: {self.id} | Nome: {self.nome} | Plano: {self.plano.nome} | Vence em: {self.data_vencimento.date()}")

    def verificar_acesso(self):
        if not self.ativo:
            raise AlunoInativoException("Aluno inativo!")
        if datetime.now() > self.data_vencimento:
            raise PlanoVencidoException("Plano vencido!")
        if self.dentro:
            raise AcessoNegadoException("Aluno já está dentro!")

class CLT:
    def __init__(self, salario: float):
        self.salario = salario

class Funcionario(Pessoa, CLT):
    def __init__(self, id: int, nome: str, idade: int, salario: float, cargo: str):
        Pessoa.__init__(self, id, nome, idade)
        CLT.__init__(self, salario)
        self.cargo = cargo

    def exibir_dados(self):
        print(f"Funcionário: {self.nome} | Cargo: {self.cargo} | Salário: {self.salario}")

alunos = []
planos = []
acessos = []
funcionarios = []

def cadastrar_plano():
    nome = input("Nome do plano: ")
    valor = float(input("Valor: "))
    duracao = int(input("Duração (dias): "))
    planos.append(Plano(nome, valor, duracao))
    print("Plano cadastrado!")

def cadastrar_aluno():
    if not planos:
        print("Cadastre um plano primeiro!")
        return

    while True:
        try:
            chosen_id = int(input("Escolha um ID para o aluno: "))
            if any(a.id == chosen_id for a in alunos):
                print(f"ID {chosen_id} já existe. Por favor, escolha outro.")
            else:
                break
        except ValueError:
            print("ID inválido. Por favor, digite um número inteiro.")

    nome = input("Nome: ")
    idade = int(input("Idade: "))
    login = input("Login: ")
    senha = input("Senha: ")
    email = input("Email: ")

    print("\nPlanos disponíveis:")
    for i, p in enumerate(planos):
        print(f"{i} - {p.nome}")

    while True:
        try:
            escolha = int(input("Escolha o plano: "))
            if 0 <= escolha < len(planos):
                plano = planos[escolha]
                break
            else:
                print("Opção de plano inválida. Escolha um número da lista.")
        except ValueError:
            print("Entrada inválida. Por favor, digite um número.")

    aluno = Aluno(chosen_id, nome, idade, login, senha, plano)
    alunos.append(aluno)

    print("Aluno cadastrado!")

def cadastrar_funcionario():
    while True:
        try:
            chosen_id = int(input("Escolha um ID para o funcionário: "))
            if any(f.id == chosen_id for f in funcionarios) or any(a.id == chosen_id for a in alunos):
                print(f"ID {chosen_id} já existe (entre funcionários ou alunos). Por favor, escolha outro.")
            else:
                break
        except ValueError:
            print("ID inválido. Por favor, digite um número inteiro.")

    nome = input("Nome do funcionário: ")
    idade = int(input("Idade: "))
    salario = float(input("Salário: "))
    cargo = input("Cargo: ")

    funcionario = Funcionario(chosen_id, nome, idade, salario, cargo)
    funcionarios.append(funcionario)
    print("Funcionário cadastrado!")

def listar_alunos():
    for a in alunos:
        a.exibir_dados()

def listar_funcionarios():
    if not funcionarios:
        print("Nenhum funcionário cadastrado.")
        return
    print("\nFuncionários cadastrados:")
    for f in funcionarios:
        f.exibir_dados()

def registrar_entrada():
    try:
        id = int(input("ID do aluno: "))
        aluno = next((a for a in alunos if a.id == id), None)

        if not aluno:
            raise Exception("Aluno não encontrado!")

        aluno.verificar_acesso()

        aluno.dentro = True
        acessos.append(Acesso(id, datetime.now(), "entrada"))

        print("Entrada liberada!")

    except (PlanoVencidoException, AlunoInativoException, AcessoNegadoException) as e:
        print(f"Erro: {e}")
    except Exception as e:
        print(f"Erro geral: {e}")
    finally:
        print("Operação finalizada.")

def registrar_saida():
    try:
        id = int(input("ID do aluno: "))
        aluno = next((a for a in alunos if a.id == id), None)

        if not aluno:
            raise Exception("Aluno não encontrado!")

        if not aluno.dentro:
            raise Exception("Aluno não está dentro!")

        aluno.dentro = False
        acessos.append(Acesso(id, datetime.now(), "saida"))

        print("Saída registrada!")

    except Exception as e:
        print(f"Erro: {e}")

def listar_acessos():
    for a in acessos:
        aluno = next((s for s in alunos if s.id == a.id_aluno), None)
        aluno_nome = aluno.nome if aluno else "Desconhecido"
        print(f"Aluno ID: {a.id_aluno} | Nome: {aluno_nome} | {a.tipo} | {a.data_hora}")

def alunos_ativos():
    ativos = [a for a in alunos if a.ativo]
    print("\nAlunos ativos:")
    for a in ativos:
        print(a.nome)

def menu():
    while True:
        print("\n===== ACADEMIA ====")
        print("1 - Cadastrar plano")
        print("2 - Cadastrar aluno")
        print("3 - Listar alunos")
        print("4 - Registrar entrada")
        print("5 - Registrar saída")
        print("6 - Listar acessos")
        print("7 - Alunos ativos")
        print("8 - Cadastrar funcionário")
        print("9 - Listar funcionários")
        print("0 - Sair")

        try:
            opcao = int(input("Escolha: "))

            match opcao:
                case 1:
                    cadastrar_plano()
                case 2:
                    cadastrar_aluno()
                case 3:
                    listar_alunos()
                case 4:
                    registrar_entrada()
                case 5:
                    registrar_saida()
                case 6:
                    listar_acessos()
                case 7:
                    alunos_ativos()
                case 8:
                    cadastrar_funcionario()
                case 9:
                    listar_funcionarios()
                case 0:
                    print("Saindo...")
                    break
                case _:
                    print("Opção inválida!")

        except ValueError:
            print("Digite um número válido!")

if __name__ == "__main__":
    menu()

Overwriting main.py


You can now run this script using the command below:

In [ ]:
!python main.py